# Foundations & SOLID Principles

## What's covered

- What design patterns are, and why a 1994 catalog is still vocabulary engineers use every day
- The Gang of Four catalog at a glance — three families, twenty-three patterns
- How Python changes the picture — patterns that dissolve into language features, and the ones that stay
- UML class diagram vocabulary used throughout this series
- The five SOLID principles, with Python and Kotlin examples for each
- When patterns help and when they get in the way


## Why design patterns exist

If you have written real applications in Java, Scala, or Kotlin, you have used design patterns — you may not have named them. Every time you passed a comparator to `sort`, you used Strategy. Every time you wrote a `for` loop over a collection, you used Iterator. Every time you injected a repository into a service through a constructor, you used Dependency Injection.

A design pattern is a **name plus an intent**. It is shared vocabulary for a recurring design problem and the standard shape engineers reach for to solve it. The vocabulary matters because design discussions otherwise drown in detail — "we should pass a lambda to invert control of the comparison" versus "use Strategy". Same idea; the second is one word.

The canonical catalog is *Design Patterns: Elements of Reusable Object-Oriented Software* by Gamma, Helm, Johnson, and Vlissides — published in 1994 and known forever as the Gang of Four. They cataloged twenty-three patterns that kept recurring in C++ and Smalltalk codebases. Three decades later, with different languages and different frameworks, most of those twenty-three still appear constantly in modern code.

A useful description of any pattern has four parts:

- **Intent** — the design problem the pattern solves, stated in one sentence
- **Structure** — the participating classes and how they relate (we will use UML for this)
- **Consequences** — what the pattern gives you, and what it takes away
- **When not to use** — the situations where the pattern adds paperwork without paying for itself


## The Gang of Four catalog at a glance

The twenty-three patterns split into three families based on the problem they address.

**Creational (5)** — patterns about *constructing* objects. They decouple the code that uses an object from the code that decides which concrete class to build.

| Pattern | One-line intent |
|---|---|
| Singleton | A class that has exactly one instance, globally accessible |
| Factory Method | A method whose subclasses choose which concrete class to instantiate |
| Abstract Factory | A factory of factories — for families of related products |
| Builder | Step-by-step construction of complex objects, with the same steps producing different shapes |
| Prototype | New objects made by cloning an existing one |

**Structural (7)** — patterns about *composing* objects into larger structures. The theme is composition over inheritance.

| Pattern | One-line intent |
|---|---|
| Adapter | Make a class with the wrong interface usable through the right one |
| Bridge | Split an abstraction from its implementation so both can vary independently |
| Composite | Treat individual objects and groups of objects uniformly |
| Decorator | Add behaviour to an object without changing its class |
| Facade | Provide a single simplified entry point to a complex subsystem |
| Flyweight | Share fine-grained objects to save memory |
| Proxy | A stand-in that controls access to another object |

**Behavioral (11)** — patterns about *distributing responsibilities* between objects.

| Pattern | One-line intent |
|---|---|
| Chain of Responsibility | Pass a request along a chain until one handler accepts it |
| Command | Wrap a request as an object so it can be queued, logged, undone |
| Iterator | Walk through a collection without exposing its internal structure |
| Mediator | Centralize how a set of objects communicate |
| Memento | Capture an object's state so it can be restored later |
| Observer | Notify many objects when one object changes |
| State | Let an object change behaviour when its internal state changes |
| Strategy | Make a family of algorithms interchangeable at runtime |
| Template Method | Define the skeleton of an algorithm, leave specific steps to subclasses |
| Visitor | Add operations to a class hierarchy without modifying the classes |

Notebooks 02 through 05 work through each pattern with intent, structure, Python and Kotlin examples, and the situations where the pattern hurts more than it helps.


## How Python changes the picture

The Gang of Four wrote against C++ and Smalltalk — languages with rigid type systems and limited support for first-class functions. Several patterns in that book exist primarily as workarounds for what those languages cannot do.

Python has first-class functions, duck typing, multiple inheritance, dynamic attribute access, modules as singletons, decorators as a language feature, and `dataclass` for value types. Several GoF patterns largely *dissolve* into Python language features:

- **Strategy** — pass a function. Done.
- **Command** — a function or a callable object.
- **Template Method** — usually a function that takes other functions as arguments.
- **Iterator** — built into the language; every class with `__iter__` is one.
- **Singleton** — a module is one. Just put the state at module level.
- **Decorator** — the `@decorator` language feature *is* the pattern.
- **Adapter** — duck typing often makes adaptation transparent.

This does not mean the patterns are obsolete. The **intent vocabulary** is still useful. Saying "we use Strategy here" still communicates more than "we pass a function" — it tells the reader *why* you pass a function, and which slot the function fills. But the **structural ceremony** — abstract base class, interface declaration, concrete implementation hierarchy — usually disappears in idiomatic Python.

Kotlin sits in the middle. It has first-class functions and a type system that supports `interface` and `sealed class`. Many GoF patterns in Kotlin live somewhere between the Python collapse and the full C++/Java ceremony — a `fun interface` plus a few small implementations is common.

The throughline for this series: **lead with intent, code in the most idiomatic shape the language allows, name the pattern explicitly so the vocabulary still works.**


## UML class diagram vocabulary

We use simple UML class diagrams to show the structure of each pattern. A few shapes carry most of the meaning.

A **class** is a box with the class name. Fields appear under the name; methods appear below the fields. A diagram showing only the boxes and their relationships, with no field or method detail, is enough for most pattern explanations.

```
+------------------+
|    ClassName     |
+------------------+
| - field: Type    |
+------------------+
| + method(): T    |
+------------------+
```

The plus sign means public; the minus sign means private. Most pattern diagrams omit visibility.

Relationships between classes are drawn as lines and arrows:

- **Inheritance** (`extends` in Kotlin, base class in Python) — a hollow triangle pointing from subclass to parent.
- **Interface implementation** (`implements` in Kotlin, `Protocol` in Python) — a hollow triangle on a dashed line, from implementer to interface.
- **Association** — a plain solid line, meaning one class knows about another.
- **Aggregation** — a hollow diamond on the owning side, meaning the owner holds a reference but does not own the lifecycle.
- **Composition** — a filled diamond on the owning side, meaning the owner owns the lifecycle.
- **Dependency** — a dashed arrow, meaning one class uses another transiently (passed as a parameter, for instance).

Where a **sequence diagram** appears — a series of method calls between objects, drawn as vertical lifelines with horizontal arrows — it will be explained inline.


## The five SOLID principles

Before we get to specific patterns, there are five general design principles every pattern leans on. Robert C. Martin gathered and named them in the early 2000s, drawing on prior work by Bertrand Meyer and Barbara Liskov. The acronym is SOLID:

- **S** — Single Responsibility Principle
- **O** — Open-Closed Principle
- **L** — Liskov Substitution Principle
- **I** — Interface Segregation Principle
- **D** — Dependency Inversion Principle

The principles are not laws. They are heuristics — defaults that pay off in most situations. Every pattern in the GoF catalog applies one or more of them. Once you can see SOLID in your code, you can usually see which pattern fits without consulting a book.

We work through each principle with a Python example and a Kotlin example. Python first; Kotlin second.


## S — Single Responsibility Principle

A class should have *one reason to change*.

Martin's later refinement makes the phrasing precise: a class should be responsible to **one actor** — one stakeholder, one role, one decision-maker. If two different actors might both demand changes to the same class, the class has two responsibilities and should be split.

A classic violation is the "god class" — a `Report` class that knows how to compute its data, format it as HTML, persist it to disk, send it by email, and log its own metrics. Five actors, five reasons to change, one tangled file.

The fix is mechanical: split each responsibility into its own class, then compose them.


### Python


In [ ]:
from dataclasses import dataclass


# violates SRP — computes, formats, and persists, all in one class
class ReportBad:
    def __init__(self, rows: list[dict]):
        self.rows = rows

    def total(self) -> float:
        return sum(r["amount"] for r in self.rows)

    def to_html(self) -> str:
        body = "".join(f"<tr><td>{r['name']}</td><td>{r['amount']}</td></tr>" for r in self.rows)
        return f"<table>{body}</table>"

    def save_to_disk(self, path: str) -> None:
        with open(path, "w") as f:
            f.write(self.to_html())


# follows SRP — three classes, one actor each
@dataclass
class Report:
    rows: list[dict]

    def total(self) -> float:
        return sum(r["amount"] for r in self.rows)


class HtmlRenderer:
    def render(self, report: Report) -> str:
        body = "".join(f"<tr><td>{r['name']}</td><td>{r['amount']}</td></tr>" for r in report.rows)
        return f"<table>{body}</table>"


class FileStore:
    def save(self, path: str, content: str) -> None:
        with open(path, "w") as f:
            f.write(content)


report = Report(rows=[{"name": "Alice", "amount": 100}, {"name": "Bob", "amount": 50}])
html = HtmlRenderer().render(report)
FileStore().save("/tmp/report.html", html)
print(report.total())  # 150


### Kotlin

```kotlin
data class Row(val name: String, val amount: Double)

class Report(val rows: List<Row>) {
    fun total(): Double = rows.sumOf { it.amount }
}

class HtmlRenderer {
    fun render(report: Report): String {
        val body = report.rows.joinToString("") {
            "<tr><td>${it.name}</td><td>${it.amount}</td></tr>"
        }
        return "<table>$body</table>"
    }
}

class FileStore {
    fun save(path: String, content: String) {
        java.io.File(path).writeText(content)
    }
}

val report = Report(listOf(Row("Alice", 100.0), Row("Bob", 50.0)))
val html = HtmlRenderer().render(report)
FileStore().save("/tmp/report.html", html)
println(report.total())  // 150.0
```


SRP is *not* "every class does exactly one thing" — that interpretation leads to absurd fragmentation. It is "every class is responsible to one stakeholder". The accounting team wants to change `total`; the design team wants to change `render`; the ops team wants to change `save`. Three actors, three classes.


## O — Open-Closed Principle

A class should be **open for extension, closed for modification**. You should be able to add new behaviour without editing existing, working code.

The motivation is risk. Every change to existing code is a chance to break something that already works and was already tested. The principle says: structure your code so that new requirements arrive as *new code*, added alongside the old, rather than as edits to the old.

The dominant technique is replacing branching on a type tag with polymorphism. An `if/elif` chain that handles three discount strategies grows to four cases when a fourth strategy arrives — every existing case sits in a file that just changed. A `DiscountStrategy` interface with four implementations adds the fourth as a new class — no existing file changes.

Strategy is the most direct application of OCP. So are Decorator and Template Method.


### Python


In [ ]:
from typing import Callable


# violates OCP — adding a new discount type means editing this function
def price_bad(cart_total: float, customer_type: str) -> float:
    if customer_type == "regular":
        return cart_total
    elif customer_type == "member":
        return cart_total * 0.9
    elif customer_type == "vip":
        return cart_total * 0.8
    else:
        raise ValueError("unknown customer type")


# follows OCP — pass the strategy in; new strategies are new functions, no edit
DiscountStrategy = Callable[[float], float]

regular: DiscountStrategy = lambda total: total
member:  DiscountStrategy = lambda total: total * 0.9
vip:     DiscountStrategy = lambda total: total * 0.8


def price(cart_total: float, strategy: DiscountStrategy) -> float:
    return strategy(cart_total)


print(price(100, regular))  # 100
print(price(100, member))   # 90.0
print(price(100, vip))      # 80.0

# adding "employee 50% off" — zero edits to price(), zero edits to existing strategies
employee: DiscountStrategy = lambda total: total * 0.5
print(price(100, employee))  # 50.0


### Kotlin

```kotlin
fun interface DiscountStrategy {
    fun apply(total: Double): Double
}

val regular = DiscountStrategy { it }
val member  = DiscountStrategy { it * 0.9 }
val vip     = DiscountStrategy { it * 0.8 }

fun price(cartTotal: Double, strategy: DiscountStrategy): Double =
    strategy.apply(cartTotal)

println(price(100.0, regular))  // 100.0
println(price(100.0, member))   // 90.0
println(price(100.0, vip))      // 80.0

val employee = DiscountStrategy { it * 0.5 }
println(price(100.0, employee))  // 50.0
```


Notice how cleanly this maps to Python's first-class functions. The "interface" is just `Callable[[float], float]` — a type alias. The "strategy" is just a function. Kotlin's `fun interface` (a single-abstract-method interface) gives you almost the same compactness, with the type system catching mistakes at compile time.


## L — Liskov Substitution Principle

A subtype must be substitutable for its base type. If your code works against a base class `T`, swapping in any subclass of `T` must not break it.

This is Barbara Liskov's 1987 definition, dressed in OOP clothes. Concretely, a subclass cannot:

- weaken the preconditions of overridden methods (cannot demand more from callers)
- strengthen the postconditions (cannot promise less to callers)
- throw exceptions the base type does not throw
- change the meaning of a method in a way that surprises callers

The famous violation is **Rectangle / Square**. A square *is-a* rectangle in geometry class. In code, if `Rectangle` exposes `set_width` and `set_height`, a `Square` cannot honour both independently — setting the width must also set the height, or the shape stops being a square. Code that holds a `Rectangle` reference, sets width to 5 and height to 3, expects an area of 15. Pass a `Square` and you get 9. The substitution broke.

The lesson: inheritance is a strong claim, not a weak one. "Is-a" in everyday language is not always "is-a" at the type level.


### Python


In [ ]:
from typing import Protocol


# violates LSP — Square cannot honour Rectangle's contract
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def set_width(self, w: float) -> None:
        self.width = w

    def set_height(self, h: float) -> None:
        self.height = h

    def area(self) -> float:
        return self.width * self.height


class SquareBad(Rectangle):
    def __init__(self, side: float):
        super().__init__(side, side)

    def set_width(self, w: float) -> None:
        self.width = w
        self.height = w  # surprise — also changed height

    def set_height(self, h: float) -> None:
        self.width = h
        self.height = h  # surprise — also changed width


def stretch(rect: Rectangle) -> float:
    rect.set_width(5)
    rect.set_height(3)
    return rect.area()


print(stretch(Rectangle(1, 1)))   # 15 — expected
print(stretch(SquareBad(1)))      # 9  — broken; substitution does not hold


# fix: a Shape protocol, no shared mutator surface
class Shape(Protocol):
    def area(self) -> float: ...


class Rect:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height


class Square:
    def __init__(self, side: float):
        self.side = side

    def area(self) -> float:
        return self.side ** 2


def total_area(shapes: list[Shape]) -> float:
    return sum(s.area() for s in shapes)


print(total_area([Rect(5, 3), Square(4)]))  # 31


### Kotlin

```kotlin
sealed interface Shape {
    fun area(): Double
}

data class Rect(val width: Double, val height: Double) : Shape {
    override fun area() = width * height
}

data class Square(val side: Double) : Shape {
    override fun area() = side * side
}

fun totalArea(shapes: List<Shape>): Double = shapes.sumOf { it.area() }

println(totalArea(listOf(Rect(5.0, 3.0), Square(4.0))))  // 31.0
```


Two things to take away from the fix. First, **prefer composition over inheritance when the "is-a" relationship is shaky** — both `Rect` and `Square` share a parent (`Shape`) rather than one inheriting from the other. Second, **immutable types sidestep most LSP traps**: the original violation was caused by mutators. Make the data immutable and the problem often evaporates.


## I — Interface Segregation Principle

Clients should not be forced to depend on methods they do not use. Many small, focused interfaces beat one fat, do-everything interface.

The classic violation is a `Worker` interface with `work()`, `eat()`, `sleep()`. A `RobotWorker` class is forced to implement `eat()` and `sleep()` even though they make no sense for a robot. The robot ends up throwing `UnsupportedOperationException` or doing nothing — either way, the type system has lied about the robot's contract.

The fix is to split the interface by role. `Workable` has `work()`. `Feedable` has `eat()`. `Sleepable` has `sleep()`. A human worker implements all three; a robot worker implements only `Workable`.

ISP is what makes Dependency Inversion (D, coming next) practical. Small interfaces are easy to inject and easy to mock in tests. Big interfaces force your tests to stub out methods you do not use.


### Python


In [ ]:
from typing import Protocol


# violates ISP — a Robot is forced to implement eat() and sleep()
class WorkerBad(Protocol):
    def work(self) -> None: ...
    def eat(self) -> None: ...
    def sleep(self) -> None: ...


# follows ISP — three small protocols by role
class Workable(Protocol):
    def work(self) -> None: ...


class Feedable(Protocol):
    def eat(self) -> None: ...


class Sleepable(Protocol):
    def sleep(self) -> None: ...


class Human:
    def work(self) -> None:  print("human working")
    def eat(self) -> None:   print("human eating")
    def sleep(self) -> None: print("human sleeping")


class Robot:
    def work(self) -> None:  print("robot working")
    # no eat, no sleep — Robot only commits to Workable


def manage_shift(workers: list[Workable]) -> None:
    for w in workers:
        w.work()


manage_shift([Human(), Robot()])


### Kotlin

```kotlin
interface Workable { fun work() }
interface Feedable { fun eat() }
interface Sleepable { fun sleep() }

class Human : Workable, Feedable, Sleepable {
    override fun work()  = println("human working")
    override fun eat()   = println("human eating")
    override fun sleep() = println("human sleeping")
}

class Robot : Workable {
    override fun work() = println("robot working")
}

fun manageShift(workers: List<Workable>) {
    workers.forEach { it.work() }
}

manageShift(listOf(Human(), Robot()))
```


Python's `Protocol` (from `typing`) implements ISP elegantly. A protocol is a *structural* type — any class with the right shape satisfies it, no inheritance required. You can declare a function parameter as `Workable` and any object with a `work()` method qualifies. Kotlin needs explicit `interface` declarations, but the principle is identical.


## D — Dependency Inversion Principle

High-level modules should not depend on low-level modules. Both should depend on abstractions.

Concretely: a `UserService` should not depend on a `PostgresUserRepo`. Both should depend on a `UserRepo` interface. The service is written against the interface; the concrete implementation is injected at runtime.

The "inversion" is the direction of the dependency arrow. Without the principle, the high-level business logic imports the low-level database code — high depends on low. With the principle, the database code implements an interface owned by the business logic — low depends on the abstraction, and so does high. The arrow flipped.

This is the principle that makes testing tractable. A `UserService` that depends directly on `PostgresUserRepo` is hard to unit-test — you need a real database. A `UserService` that depends on `UserRepo` is easy — you pass in an in-memory fake.

DIP is the bridge to **dependency injection**, the topic of notebook 06. The principle says *what* to do; dependency injection is *how* — constructor injection, framework containers, the rest.


### Python


In [ ]:
from typing import Protocol


# violates DIP — UserService directly depends on the concrete PostgresUserRepo
class PostgresUserRepoBad:
    def find(self, user_id: int) -> dict:
        # imagine a real DB call here
        return {"id": user_id, "name": "Alice"}


class UserServiceBad:
    def __init__(self):
        self.repo = PostgresUserRepoBad()  # baked-in concrete dependency

    def greet(self, user_id: int) -> str:
        return f"Hello, {self.repo.find(user_id)['name']}"


# follows DIP — depend on the UserRepo protocol; inject the concrete repo
class UserRepo(Protocol):
    def find(self, user_id: int) -> dict: ...


class PostgresUserRepo:
    def find(self, user_id: int) -> dict:
        return {"id": user_id, "name": "Alice"}


class InMemoryUserRepo:
    def __init__(self, users: dict[int, dict]):
        self.users = users

    def find(self, user_id: int) -> dict:
        return self.users[user_id]


class UserService:
    def __init__(self, repo: UserRepo):
        self.repo = repo

    def greet(self, user_id: int) -> str:
        return f"Hello, {self.repo.find(user_id)['name']}"


# production: real database
prod = UserService(PostgresUserRepo())

# tests: in-memory fake, no database needed
fake_repo = InMemoryUserRepo({1: {"id": 1, "name": "Bob"}})
test = UserService(fake_repo)
print(test.greet(1))  # Hello, Bob


### Kotlin

```kotlin
data class User(val id: Int, val name: String)

interface UserRepo {
    fun find(userId: Int): User
}

class PostgresUserRepo : UserRepo {
    override fun find(userId: Int) = User(userId, "Alice")
}

class InMemoryUserRepo(private val users: Map<Int, User>) : UserRepo {
    override fun find(userId: Int) = users.getValue(userId)
}

class UserService(private val repo: UserRepo) {
    fun greet(userId: Int) = "Hello, ${repo.find(userId).name}"
}

// production
val prod = UserService(PostgresUserRepo())

// tests
val fakeRepo = InMemoryUserRepo(mapOf(1 to User(1, "Bob")))
val test = UserService(fakeRepo)
println(test.greet(1))  // Hello, Bob
```


DIP is the most load-bearing of the five principles for testable, maintainable systems. A codebase that consistently inverts dependencies can swap any layer — database, message bus, payment gateway — for a fake in tests and a different implementation in production. A codebase that does not, pays for it forever.


## When patterns help and when they get in the way

Three signs you should reach for a pattern:

- **You have seen the same shape twice.** Once is a special case; twice starts to look like a pattern. Three times demands one.
- **Naming the design clarifies the conversation.** If "I want to add a step in the pipeline" and "I want to wrap a behaviour" produce different code reviews, the vocabulary is earning its keep.
- **The pattern's intent matches your problem.** Not the structure — the intent. Strategy fits when algorithms are interchangeable, not when you happen to have several classes with one method each.

Three signs the pattern is hurting:

- **The pattern's structure is bigger than the problem.** A `SingletonFactoryBuilder` to construct one logger is paperwork dressed as architecture.
- **The pattern was applied before the problem appeared.** Speculative patterns ("we might need to swap implementations later") usually rot before the swap arrives.
- **The pattern obscures the domain.** If a reader has to recognize a pattern before they can read the business logic, the pattern is in the way.

The honest rule of thumb: **start with the simplest code that works, refactor toward a pattern when a smell appears, delete the pattern when the smell goes away.** Patterns are tools, not goals.


Notebook two turns to **Creational patterns** — the family that decouples *what* gets constructed from *who* constructs it. Five patterns: Singleton, Factory Method, Abstract Factory, Builder, and Prototype. We will look at the intent of each, the structure, a Python example, a Kotlin example, and the situations where the pattern is the wrong choice.
